> **수정 설계 · 현재 단계는 출처 파일럿입니다.**
> 목표 기간: **2017-01-01~2026-10-25**. 먼저 `midterm_source_pilot.ipynb`과 `SOURCE_PILOT_REVIEW.md`를 확인하세요.
> 아래는 기존 **뉴스 단독** 분석 경로입니다. Reddit 자료를 합치지 않습니다.
> 주 지수는 `WSI_count`, 연속 지수는 강건성 비교입니다. 대규모 수집은 파일럿 검토 전 중단합니다.

# Validating the Fear & Greed Index with Web-Based Financial Sentiment
### Correlation and Lead-Lag Analysis
### 중간 페이퍼 준비용 분석 노트북

**Web Data Scraping and Text Analysis · 코드와 분석 산출물 정리**

원자료를 수집하고 표·그림을 만든다. 연구 동기, 행동재무 해석, 최종 보고서는 별도로 작성한다.
족보 PDF는 구성만 참고하며 실증 데이터에 포함하지 않는다.

| 연구 질문 | 산출물 |
|---|---|
| RQ1 · Fear/Greed 언어 차이 | 빈도표, 국면별 워드클라우드, TF-IDF |
| RQ2 · 일별 웹 감성 구성 | VADER 네 점수, 세 가지 WSI, 문서 수 |
| RQ3 · FGI와의 동행 | 시계열, z-score, 산점도, Pearson/Spearman |
| RQ4 · 선행·후행 연관성 | 달력 날짜 ±7일 시차표, 단어별 시점 |

> **실제 자료 상태:** 구현 중 GDELT가 HTTP 429를 반환해 전체 뉴스 수집은 확인되지 않았다.
> 아래 셀은 실제 자료를 확보한 뒤 계산한다. 실행하지 않은 결과를 넣지 않았다.

**순서:** 환경 → 수집 → 전처리 → VADER → 일별 지수 → 국면별 언어 → 상관·시차 → 그림


## 1. 환경과 공통 설정

현재 커널의 Python에 `requirements.txt`를 설치한다. 필요한 경우 `%pip` 줄의 주석을 해제한다.
저장소 루트 또는 `midterm paper` 폴더에서 실행할 수 있다.
계산은 번호별 `.py` 파일을 직접 호출하므로 코드 두 벌을 수정할 필요가 없다.


In [ ]:
from pathlib import Path
import runpy
import sys

BASE = Path.cwd().resolve()
if not (BASE / "midterm paper.py").exists():
    BASE = BASE / "midterm paper"
if not (BASE / "midterm paper.py").exists():
    raise FileNotFoundError("노트북을 프로젝트 폴더 또는 저장소 루트에서 실행하세요.")
print("Python:", sys.executable)
print("Project:", BASE)
if (BASE / ".packages").is_dir():
    sys.path.insert(0, str(BASE / ".packages"))
# 필요한 경우 실행한 뒤 커널을 다시 시작하세요.
# %pip install -r "{BASE / 'requirements.txt'}"


In [ ]:
import importlib.util
required = ["pandas", "numpy", "requests", "bs4", "vaderSentiment", "scipy", "sklearn", "matplotlib", "wordcloud"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise ImportError(f"현재 커널에 패키지가 없습니다: {missing}. 위 셀에서 requirements.txt를 설치하세요.")

import pandas as pd
from IPython.display import display, Image, Markdown
pd.set_option("display.max_columns", 18)
pd.set_option("display.max_colwidth", 100)
pipeline = runpy.run_path(str(BASE / "midterm paper.py"))
Config = pipeline["Config"]
run_stage = pipeline["run_stage"]
read_csv = pipeline["common"].read_csv


### 여기에서만 연구 설정 바꾸기

기본 기간은 **2017-01-01 ~ 2026-10-25**, 날짜 경계는 뉴욕 시간이다.
DOC API의 최근 약 3개월 범위를 벗어나면 기존 원자료 또는 로컬 CSV가 필요하다.
발행 시각이 필요한 설계라면 실제 발행 시각이 있는 CSV를 `text_input`에 지정한다.
기간·검색 조건을 상관이 높아지도록 사후 조정하지 않는다.


In [ ]:
# CONFIGURATION
cfg = Config(
    root=BASE,
    start="2017-01-01",
    end="2026-10-25",
    timezone="America/New_York",
    min_daily_docs=20,
    max_lag=7,
    top_n=30,
    timing_terms_n=15,
    fgi_source="archive",  # CNN 주식시장 FGI의 제3자 아카이브
    text_input=None,        # 예: str(BASE / "data/input/news.csv")
    fgi_input=None,         # 예: str(BASE / "data/input/fgi.csv")
    fgi_source_url="",     # FGI 로컬 입력을 쓸 때 실제 출처 URL 필수
)
cfg.setup()
print(f"{cfg.start} ~ {cfg.end} | {cfg.timezone} | 일별 최소 {cfg.min_daily_docs}건")
print("검색 조건:", cfg.query)


## 2. 실제 자료 수집 또는 가져오기

FGI부터 받은 뒤 뉴스 헤드라인을 받는다. 성공한 날짜의 JSON은 재사용한다.
401/403/418/429, robots 불허, 잘못된 JSON에서는 오류를 표시하고 중단한다.
요청 실패를 뉴스 0건으로 처리하거나 차단을 우회하지 않는다.

**GDELT `seendate`는 최초 탐지 시각**이다. `observed_at`에 보관하고 발행 시각을 만들어 넣지 않는다.
UTC 날짜당 최대 250건의 날짜 내림차순 응답이므로 전수·무작위 표본이 아니다.
FGI는 CNN **주식시장** 지수이며 암호화폐 지수로 바꾸지 않는다.


In [ ]:
fgi = run_stage(cfg, "collect-fgi")
display(fgi.head())
display(fgi.tail())


In [ ]:
raw = run_stage(cfg, "collect-news")
print("수집/가져온 원문 수:", f"{len(raw):,}")
display(raw[["published_at", "raw_text", "source", "url"]].head())
manifest_path = cfg.results / "collection_manifest.csv"
if manifest_path.exists():
    manifest = read_csv(manifest_path)
    print("응답 상한 도달 UTC 날짜:", int(manifest.cap_hit.sum()))
    display(manifest[["date_utc", "n_returned", "cap_hit"]].head(10))


## 3. 원문 보존과 전처리

`raw_text`는 보존한다. 감성 입력은 HTML·공백만 정리해 대문자/문장부호/부정을 유지한다.
빈도 입력은 별도 토큰화를 적용한다. 금융 은어·밈·욕설을 자동 삭제하지 않는다.
동일 URL을 제거한 뒤 같은 날의 동일 정규화 헤드라인을 제거한다.
시간대 없는 입력 시각, 필수 열 누락, 빈 텍스트는 오류로 확인한다.


In [ ]:
processed = run_stage(cfg, "preprocess")
display(processed[["date", "timestamp_kind", "raw_text", "sentiment_text", "frequency_text"]].head(8))
display(read_csv(cfg.results / "daily_document_count.csv").describe())


## 4. VADER 문서별 감성

positive: compound ≥ 0.05 / negative: compound ≤ −0.05 / neutral: 그 사이.
기본 VADER를 그대로 사용하고 FGI와 일치하도록 사전·점수를 수정하지 않는다.
왜도·첨도는 분포 통계이며 투자자 성향의 직접 증거가 아니다.


In [ ]:
scored = run_stage(cfg, "sentiment")
display(scored[["raw_text", "negative", "neutral", "positive", "compound", "sentiment_label"]].head(10))
display(read_csv(cfg.results / "sentiment_descriptive_statistics.csv").T.rename(columns={0: "value"}))


## 5. 일별 감성지수와 FGI 결합

| 지수 | 계산 | 중립 처리 |
|---|---|---|
| **WSI_continuous (강건성)** | 50 × (일평균 compound + 1) | 포함 |
| **WSI_count (주 지수)** | 50 × {1 + (양성 수 − 음성 수)/전체 수} | 분모에 포함 |
| WSI_binary | 100 × 양성 수/(양성 수 + 음성 수) | 제외 |

모두 중립인 날은 binary가 결측이다. 뉴스 없는 날은 모든 지수가 결측이다.
FGI 없는 날은 채우지 않고 국면 비교에서 제외한다. 기본 일별 최소 20건이다.

Bernoulli 정규근사는 **일별 추정 비율의 표집분포**에 관한 것이다.
독립성 가정 아래 표준오차·Wilson 구간을 저장한다. 재배포 기사 등 문서 간 의존성은 남을 수 있다.
원문 감성 관측값 자체가 정규분포가 된다는 뜻이 아니다.


In [ ]:
merged = run_stage(cfg, "daily")
display(merged[["date", "n_docs", "mean_sentiment", "WSI_continuous", "WSI_count", "WSI_binary", "fgi", "regime", "analysis_eligible"]].head(15))
display(read_csv(cfg.results / "regime_summary.csv"))
import json
print(json.dumps(json.loads((cfg.results / "merge_diagnostics.json").read_text(encoding="utf-8")), ensure_ascii=False, indent=2))


## 6. 국면별 언어와 단어 시점

국면은 원자료 라벨을 우선 사용한다. 없는 경우에만 `regime_basis`에 표시한 연구자 구간을 쓴다.
5개 국면 빈도표, Extreme을 포함한 Fear/Greed 비교, 공통 어휘집의 평균 문서 TF-IDF를 계산한다.

단어 시점은 **FGI 상관을 보기 전에** 전체 문서빈도 상위 15개를 선택한다.
`word_daily_patterns.csv`에는 토큰 수와 그 단어가 든 문서 비율을 구분해 저장한다.
국면별 문서 비율은 `word_regime_patterns.csv`에서 확인한다.


In [ ]:
frequencies = run_stage(cfg, "words")
display(frequencies.head(30))
display(read_csv(cfg.results / "regime_tfidf.csv").groupby("regime").head(8))
display(read_csv(cfg.results / "selected_timing_terms.csv"))


### 문맥 검토용 자료

`regime_disagreement_candidates.csv`는 FGI 국면과 VADER 부호가 반대인 후보이다.
**국면과 다르다는 이유만으로 모델 오류라고 판정하지 않는다.**
별도 무작위 `human_review_template.csv`의 빈 열에 사람이 의미·은어·반어를 기록할 수 있다.
원래 감성점수는 변경하지 않는다.


In [ ]:
candidates = read_csv(cfg.results / "regime_disagreement_candidates.csv")
display(candidates.reindex(columns=["date", "regime", "raw_text", "compound", "review_reason"]).head(15))
print("수동 검토 표본:", cfg.results / "human_review_template.csv")


## 7. 상관과 달력 날짜 시차

**ρ(k) = Corr(WSIₜ, FGIₜ₊ₖ)**

- k > 0: 오늘 웹 감성과 미래 FGI 비교. 웹 감성이 먼저 움직이는 연관성 후보.
- k < 0: 오늘 웹 감성과 과거 FGI 비교. FGI가 먼저 움직이는 연관성 후보.

달력 날짜를 맞춘 후 결측 쌍을 제외한다. FGI 휴장일을 지우고 행을 이동하지 않는다.
표본 수, Pearson, Spearman을 저장하며 수준/달력 1일 차분/문서 수 기준 민감도를 확인한다.
여러 단어·시차를 탐색하므로 최대 상관 하나로 결론 내리지 않는다.
자기상관을 무시한 유의확률, 인과 또는 검증된 예측 성능을 출력하지 않는다.
FGI 마감 시각과 하루 전체 뉴스의 차이도 별도로 고려한다.


In [ ]:
lag_results = run_stage(cfg, "association")
display(read_csv(cfg.results / "correlations.csv"))
display(lag_results.loc[lag_results["index"].eq("WSI_count") & lag_results.specification.eq("levels")])
display(read_csv(cfg.results / "correlation_sensitivity.csv"))


## 8. 그림 모아보기

그림을 `figures/`에 저장하고 순서대로 표시한다. 특정 국면의 자료가 없으면 관측 없음으로 표시한다.
실제 해석은 표본 규모·누락·수집 상한 진단과 함께 별도로 작성한다.


In [ ]:
artifacts = run_stage(cfg, "figures")
for item in artifacts:
    display(Markdown(f"**{item['file']}** · {item['caption']}"))
    display(Image(filename=str(cfg.figures / item["file"]), width=1000))


## 9. 저장 파일 확인

원문/처리 자료/결과/그림을 구분한다. 출처와 재실행 명령은 `README.md`에 정리했다.
이 노트북은 보고서 문장이나 연구 결론을 생성하지 않는다.


In [ ]:
for folder in [cfg.raw, cfg.processed, cfg.results, cfg.figures]:
    files = sorted(p for p in folder.iterdir() if p.is_file())
    print(f"\n{folder.relative_to(cfg.root)}/ — {len(files)} files")
    for file in files:
        print(" ", file.name)
